# Universidad Peruana Cayetano Heredia (UPCH)
## Pipeline 2: Modelo Predictivo con Regresion Lineal Multiple
### Caso: Mineria de Asteroides - Tipo A
**Estudiante:** Leonardo Javier Nunton Fajardo | **Docente:** Yannick Patrick Carrasco Merma

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---


## 1. Tarea de Machine Learning Seleccionada

**Tarea:** Regresion supervisada

**Problema:** Dado un asteroide recien descubierto, queremos predecir su
**valor economico neto (millones de USD)** usando caracteristicas observables
desde la Tierra, sin enviar una mision exploratoria.

### Modelo elegido: Regresion Lineal Multiple

Se eligio la **Regresion Lineal Multiple** por las siguientes razones:

1. **Interpretabilidad:** Cada coeficiente explica cuanto aporta cada variable al valor predicho.
2. **Adecuado al problema:** El valor mineral tiene relacion lineal con el diametro y el tipo espectral.
3. **Nivel del curso:** Es el modelo estandar de regresion en introduccion a Machine Learning.
4. **Estabilidad:** Con 300 muestras, la regresion lineal es estable y no se sobreajusta.

Ecuacion del modelo:

    y = b0 + b1*diametro + b2*albedo + b3*delta_v + b4*tipo_S + b5*tipo_M


## 2. Dataset Sintetico del Caso


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("Librerias cargadas.")


In [ ]:
# Generamos el dataset sintetico basado en el caso de Mineria de Asteroides
np.random.seed(42)
n = 300

# Tipo espectral: C (rocoso), S (siliceo), M (metalico)
tipos = np.random.choice(["C", "S", "M"], size=n, p=[0.6, 0.3, 0.1])

# Diametro en km (log-normal: asteroides pequenos son mas comunes)
diametro = np.clip(np.round(np.random.lognormal(0.2, 0.8, n), 2), 0.05, 12.0)

# Albedo segun tipo espectral
albedo = []
for t in tipos:
    if t == "C":
        albedo.append(round(np.random.uniform(0.04, 0.10), 4))
    elif t == "S":
        albedo.append(round(np.random.uniform(0.15, 0.28), 4))
    else:
        albedo.append(round(np.random.uniform(0.10, 0.22), 4))
albedo = np.array(albedo)

# Delta-V requerido en km/s
delta_v = np.clip(np.round(np.random.normal(5.8, 1.4, n), 2), 3.8, 10.5)

# Valor neto estimado (variable objetivo)
# Los tipo M valen mucho mas; diametro alto y delta_v bajo aumentan el valor
base_valor = np.array([480.0 if t == "M" else (160.0 if t == "S" else 60.0) for t in tipos])
valor_neto = np.round(
    base_valor * (diametro ** 0.8) * (1 + albedo * 2) - delta_v * 35 + np.random.normal(0, 30, n),
    2
)

df = pd.DataFrame({
    "diametro_km": diametro,
    "albedo": albedo,
    "delta_v": delta_v,
    "tipo_espectral": tipos,
    "valor_neto_m_usd": valor_neto
})

print(f"Dataset creado con {len(df)} asteroides sinteticos.")
display(df.head())


In [ ]:
print("Estadisticas basicas del dataset:")
display(df.describe())
print("\nCantidad por tipo espectral:")
print(df["tipo_espectral"].value_counts())


### Analisis:
El dataset tiene **300 asteroides** con sus caracteristicas observables.
La variable objetivo `valor_neto_m_usd` tiene alta variabilidad:
los tipo M (metalicos) valen mucho mas que los tipo C (rocosos).


## 3. Exploracion Visual del Dataset


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histograma del target
axes[0].hist(df["valor_neto_m_usd"], bins=25, color="steelblue", edgecolor="black")
axes[0].set_title("Distribucion del Valor Neto")
axes[0].set_xlabel("Valor Neto (M USD)")
axes[0].set_ylabel("Frecuencia")

# Diametro vs Valor segun tipo espectral
colores_tipo = {"C": "gray", "S": "orange", "M": "gold"}
for tipo, grupo in df.groupby("tipo_espectral"):
    axes[1].scatter(grupo["diametro_km"], grupo["valor_neto_m_usd"],
                    label=f"Tipo {tipo}", alpha=0.6, color=colores_tipo[tipo])
axes[1].set_title("Diametro vs Valor Neto")
axes[1].set_xlabel("Diametro (km)")
axes[1].set_ylabel("Valor Neto (M USD)")
axes[1].legend()

# Delta-V vs Valor
axes[2].scatter(df["delta_v"], df["valor_neto_m_usd"], alpha=0.4, color="tomato")
axes[2].set_title("Delta-V vs Valor Neto")
axes[2].set_xlabel("Delta-V (km/s)")
axes[2].set_ylabel("Valor Neto (M USD)")

plt.tight_layout()
plt.show()


### Analisis:
- Los **tipo M** (metalicos, amarillo) tienen el mayor valor aunque son pocos.
- A mayor **diametro**, mayor valor (relacion positiva visible).
- A mayor **delta_v**, menor valor neto (mas combustible = menor ganancia).


## 4. Preprocesamiento de los Datos

Orden correcto de ML:
1. Primero dividir en entrenamiento y prueba
2. Despues escalar (para no filtrar informacion del test set al scaler)


In [ ]:
# One-Hot Encoding del tipo espectral
df_model = pd.get_dummies(df, columns=["tipo_espectral"], drop_first=True)

# Separamos X (features) e y (target)
X = df_model.drop(columns=["valor_neto_m_usd"])
y = df_model["valor_neto_m_usd"]

print("Variables de entrada (X):", list(X.columns))
print("Variable objetivo (y):", y.name)

# Division: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\nEntrenamiento: {len(X_train)} muestras")
print(f"Prueba:        {len(X_test)} muestras")


In [ ]:
# Escalado estandar: media=0, desviacion=1
# IMPORTANTE: solo fit() con entrenamiento; transform() para ambos conjuntos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print("Escalado aplicado correctamente.")


## 5. Entrenamiento del Modelo


In [ ]:
# Creamos y entrenamos el modelo de Regresion Lineal Multiple
modelo = LinearRegression()
modelo.fit(X_train_scaled, y_train)

# Predicciones en el conjunto de prueba
y_pred = modelo.predict(X_test_scaled)

print("Modelo entrenado.")
print(f"\nIntercept (b0): {modelo.intercept_:.2f}")
print("\nCoeficientes:")
for nombre, coef in zip(X.columns, modelo.coef_):
    print(f"  {nombre:<28} = {coef:.4f}")


### Interpretacion de los coeficientes:
- Coeficiente **positivo** --> la variable aumenta el valor predicho.
- `tipo_espectral_M` tiene el mayor valor: los metalicos son los mas valiosos.
- `delta_v` tiene coeficiente **negativo**: mas combustible = menor ganancia neta.


## 6. Evaluacion del Modelo


In [ ]:
r2   = r2_score(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae  = mean_absolute_error(y_test, y_pred)

print("=" * 42)
print("       METRICAS DE EVALUACION")
print("=" * 42)
print(f"  R2  (coef. determinacion): {r2:.4f}")
print(f"  RMSE (error cuadratico):   {rmse:.2f} M USD")
print(f"  MAE  (error absoluto):     {mae:.2f} M USD")
print("=" * 42)

if r2 > 0.80:
    print("\n OK R2 > 0.80: el modelo explica bien la variabilidad del valor.")
elif r2 > 0.60:
    print("\n R2 entre 0.60-0.80: ajuste aceptable.")
else:
    print("\n R2 menor a 0.60: el modelo necesita mejoras.")


## 7. Graficos Interpretativos


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Grafico 1: Valores Reales vs Predichos
axes[0].scatter(y_test, y_pred, alpha=0.6, color="steelblue")
lim_min = min(y_test.min(), y_pred.min())
lim_max = max(y_test.max(), y_pred.max())
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], "r--", label="Prediccion perfecta")
axes[0].set_title("Valores Reales vs Predichos")
axes[0].set_xlabel("Valor Real (M USD)")
axes[0].set_ylabel("Valor Predicho (M USD)")
axes[0].legend()
axes[0].text(0.05, 0.92, f"R2 = {r2:.3f}", transform=axes[0].transAxes,
             fontsize=11, color="darkblue", fontweight="bold")

# Grafico 2: Coeficientes del modelo
coefs = list(modelo.coef_)
nombres = list(X.columns)
colores = ["green" if c > 0 else "tomato" for c in coefs]

axes[1].barh(nombres, coefs, color=colores)
axes[1].axvline(x=0, color="black", linewidth=0.8)
axes[1].set_title("Coeficientes del Modelo Lineal")
axes[1].set_xlabel("Valor del coeficiente")
axes[1].set_ylabel("Variable")

plt.tight_layout()
plt.show()


### Interpretacion de los graficos:
1. **Reales vs Predichos:** Los puntos cerca de la linea roja = buena prediccion.
2. **Coeficientes:** Barras verdes = variables que aumentan el valor;
   barras rojas = variables que lo disminuyen.
   El tipo M y el diametro son los predictores mas importantes.


## 8. Prediccion en Nuevos Asteroides


In [ ]:
# Nuevos asteroides a evaluar
nuevos = pd.DataFrame([
    # Grande, metalico, baja delta_v (muy prometedor)
    {"diametro_km": 2.1, "albedo": 0.18, "delta_v": 4.9,
     "tipo_espectral_M": 1, "tipo_espectral_S": 0},
    # Pequeno, rocoso, alta delta_v (poco prometedor)
    {"diametro_km": 0.3, "albedo": 0.06, "delta_v": 9.2,
     "tipo_espectral_M": 0, "tipo_espectral_S": 0},
    # Mediano, siliceo, delta_v moderada
    {"diametro_km": 1.0, "albedo": 0.20, "delta_v": 5.5,
     "tipo_espectral_M": 0, "tipo_espectral_S": 1},
])

# Alineamos columnas con las del entrenamiento
for col in X.columns:
    if col not in nuevos.columns:
        nuevos[col] = 0
nuevos = nuevos[X.columns]

nuevos_scaled = scaler.transform(nuevos)
predicciones  = modelo.predict(nuevos_scaled)

nuevos["Valor_Predicho_M_USD"] = predicciones.round(2)
nuevos["Recomendacion"] = ["PRIORITARIO" if v > 300 else "DESCARTAR" for v in predicciones]

print("Prediccion para nuevos asteroides:\n")
display(nuevos[["diametro_km", "delta_v", "Valor_Predicho_M_USD", "Recomendacion"]])


## Conclusiones

1. Se implemento un **pipeline completo de Machine Learning** con Regresion Lineal Multiple
   para predecir el valor economico de asteroides a partir de datos observables.

2. Las variables mas importantes fueron el **tipo espectral M** y el **diametro**,
   lo que coincide con la intuicion del dominio: los asteroides metalicos grandes son los mas valiosos.

3. La variable `delta_v` tiene impacto **negativo**: cuanto mas combustible requiere la mision,
   menor es el retorno economico neto esperado.

4. El pipeline sigue las buenas practicas de ML:
   - Division train/test **antes** del escalado
   - Evaluacion con R2, RMSE y MAE
   - Graficos interpretativos (Reales vs Predichos y coeficientes)

5. El modelo puede usarse para **priorizar misiones**: si el valor predicho supera los 300 M USD,
   el asteroide es candidato para una exploracion completa.
